# Sesión 2 — Práctica: RAG y Diseño de Caso de Uso

Esta práctica tiene **dos partes**:

**Parte 1 — Técnica (individual, ~20 min)**  
Construid vuestro propio pipeline RAG paso a paso, con el backend de vuestra elección (Gemini o Ollama).

**Parte 2 — Estratégica (grupal, ~30 min)**  
Diseñad un caso de uso de IA generativa para un proceso de vuestro sector. Presentación de 3 minutos.

---

> Las celdas marcadas con `# TODO` son las que debéis completar.  
> Las celdas de soporte ya están listas para ejecutar.

## Setup

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

### Configuración del modelo

Cambiad `BACKEND` para usar Gemini (Colab) u Ollama (local). **Solo esta celda cambia entre backends.**

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "gemini"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3.5:2b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash + gemini-embedding-001")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama, OllamaEmbeddings
    llm = ChatOllama(model=OLLAMA_MODEL)
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL} + {OLLAMA_EMBEDDING_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}")

### Carga de datos

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Cargando 200 reviews de Amazon Electronics...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(200))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

print(f"✓ {len(df)} reviews listas")
print(df[["rating", "title"]].head(5))

---

## Parte 1: Construcción del Pipeline RAG

Vais a construir el pipeline paso a paso. En la demo habéis visto el resultado final; aquí construís cada pieza.

### Ejercicio 1: Crear los documentos

Convierte el dataframe en una lista de objetos `Document` de LangChain.

Cada documento debe tener:
- `page_content`: el texto de la review (podéis incluir el título)
- `metadata`: al menos `rating` y `parent_asin`

```python
from langchain_core.documents import Document
```

In [ ]:
from langchain_core.documents import Document

# TODO: construye la lista de Documents a partir de df
docs = []

# Verificación
print(f"Documentos creados: {len(docs)}")
if docs:
    print(f"\nPrimer documento:")
    print(f"  Content (primeros 200 chars): {docs[0].page_content[:200]}")
    print(f"  Metadata: {docs[0].metadata}")

### Ejercicio 2: Construir el vector store

Usa `Chroma.from_documents()` para indexar los documentos con el modelo de embeddings configurado.

```python
from langchain_chroma import Chroma
```

Después, crea un `retriever` que devuelva los **4 documentos más relevantes** para cada consulta.

In [ ]:
from langchain_chroma import Chroma

# TODO: crear el vector store e indexar los documentos
vectorstore = None

# TODO: crear el retriever con k=4
retriever = None

# Verificación
if vectorstore is not None:
    print(f"✓ Vector store con {vectorstore._collection.count()} vectores")

if retriever is not None:
    test_docs = retriever.invoke("auriculares bluetooth")
    print(f"✓ Retriever devuelve {len(test_docs)} documentos para 'auriculares bluetooth'")

### Ejercicio 3: Construir la cadena RAG

Construye la cadena completa usando LangChain Expression Language (LCEL).

La cadena debe:
1. Recuperar documentos con el retriever
2. Formatearlos como texto
3. Inyectarlos en un prompt junto con la pregunta
4. Pasar el prompt al LLM
5. Devolver la respuesta como string

El prompt debe instruir al modelo a responder **solo** con la información de las reviews recuperadas.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# TODO: define la función format_docs que convierte una lista de Documents en un string
def format_docs(docs):
    pass

# TODO: define el prompt_rag usando ChatPromptTemplate
prompt_rag = None

# TODO: construye rag_chain usando LCEL (operador |)
rag_chain = None

print("✓ Cadena RAG lista" if rag_chain is not None else "⚠ Cadena RAG no construida aún")

### Ejercicio 4: Prueba y comparación

Prueba el pipeline con al menos dos preguntas. Para cada una, compara:
- La respuesta directa del LLM (sin RAG)
- La respuesta de tu cadena RAG

**Preguntas sugeridas** (o usad las vuestras):
- `"¿Qué problemas tienen los altavoces portátiles según los clientes?"`
- `"¿Los clientes están satisfechos con la calidad de imagen de las cámaras?"`
- `"¿Qué tipo de cables o accesorios tienen peor valoración?"`

In [ ]:
# TODO: elige una pregunta y compara respuesta directa vs. RAG
mi_pregunta = "..."

# Respuesta directa
# ...

# Respuesta RAG
# ...

### Ejercicio 5 (bonus): Filtrado por metadata

ChromaDB permite filtrar los documentos recuperados por metadata antes de calcular similitud semántica.

Crea un retriever que solo considere reviews con **rating ≤ 2** (reviews negativas).  
Pista: usa el parámetro `filter` en `as_retriever()`.

¿Cambia la calidad de las respuestas cuando solo recuperas reviews negativas?

In [ ]:
# TODO (bonus): retriever filtrado por rating <= 2
retriever_negativo = None

# Si lo has construido, pruébalo con:
# pregunta_negativa = "¿Qué falló en estos productos?"
# ...

---

## Parte 2: Diseño de Caso de Uso

**Grupos de 3–4 personas. 30 minutos.**

Rellenad las celdas de abajo con vuestro diseño. Al terminar, cada grupo presenta durante **3 minutos**.

Elegid un proceso de negocio **de vuestro sector o empresa** que podría mejorar con IA generativa.  
No tiene que ser perfecto — el objetivo es razonar en voz alta sobre viabilidad y riesgos.

### Caso de uso

**Nombre del caso de uso:**  
*(ej: "Asistente de respuesta a quejas de clientes")*

Calcular la pólisa  de seguro a través de una consulta

---

**Función de negocio:**  
*(Marketing / Ventas / Finanzas / Legal / Atención al cliente / Operaciones / Otra)*

Ventas

---

**Descripción del problema actual:**  
*(¿Qué proceso existe hoy? ¿Cuánto tiempo tarda? ¿Qué cuesta? ¿Dónde falla?)*

Minimizar el tiempo que invierten en definir el tipo de poliza para cada cliente según tipología

---

**Propuesta de solución:**  
*(¿Qué hace el sistema GenAI? ¿Qué entra, qué sale?)*

Se le cargan todos los modelos actuales para pólizas, el médico envía un prompt con los datos del paciente y el output es el modelo de póliza óptimo para ese paciente

### Arquitectura propuesta

**Estrategia principal:**  
*(Solo prompting / RAG / Agente / Fine-tuning / Combinación)*

RAG (Retrieval-Augmented Generation). El catálogo de pólizas se indexa en un vector store; cuando el médico describe al paciente, el sistema recupera las pólizas más relevantes y el LLM elige la óptima con justificación.

---

**Fuentes de datos:**  
Documentos de pólizas actuales (PDF o Word internos), tablas de primas, condiciones y exclusiones. Vienen del equipo de producto/actuarial. Actualización cada vez que se modifica el catálogo — baja frecuencia, probablemente mensual o bajo demanda.

---

**Modelo y proveedor:**  
*(Gemini / OpenAI / Ollama local / Otro — ¿por qué ese?)*

OpenAI (GPT-4o) o Gemini 1.5 Pro. Ambos son sólidos para razonamiento estructurado sobre documentos. Si hay restricciones de confidencialidad de datos de pacientes, valorar Azure OpenAI (datos en región europea) o un modelo local con Ollama.

---

**¿Build o Buy?**  
*(¿Lo construís vosotros o usáis un producto SaaS? ¿Por qué?)*

Buy/SaaS en una primera fase — usar una plataforma como Langchain + OpenAI o una herramienta no-code tipo Dify o Flowise para reducir tiempo al MVP. Si escala, se puede pasar a build propio.

### Estimación de impacto

**Impacto esperado en tiempo / coste:**  
*(Ser concretos: "de 2 horas a 15 minutos por caso", "ahorro estimado de X€/mes")*

De 20–30 minutos por paciente a menos de 2 minutos. Si se procesan 50 consultas/día, el ahorro estimado es de ~1.300 horas/año de trabajo del asesor.

---

**Métricas de éxito:**  
*(¿Cómo sabréis si funciona? ¿Qué medís?)*

% de acierto de la recomendación validada por el médico (objetivo: >85%), tiempo medio de selección de póliza, tasa de aceptación del cliente, número de consultas procesadas sin intervención manual.

---

**¿Necesita supervisión humana?**  
*(¿Para qué casos? ¿Cómo se diseña el human-in-the-loop?)*

Sí, siempre. El sistema recomienda, el médico/asesor confirma antes de presentar al cliente. Casos que siempre requieren revisión: pacientes con múltiples patologías graves, pólizas de alto valor, o cuando el sistema devuelve confianza baja.

### Roadmap a 3 meses

| Mes | Hito |
|-----|------|
| Mes 1 | *[Completad aquí]* |
| Mes 2 | *[Completad aquí]* |
| Mes 3 | *[Completad aquí]* |

### Riesgos y mitigaciones

| Riesgo | Probabilidad | Impacto | Mitigación |
|--------|-------------|---------|------------|
| *[Completad aquí]* | Alta / Media / Baja | Alto / Medio / Bajo | *[Completad aquí]* |
| *[Completad aquí]* | | | |
| *[Completad aquí]* | | | |

---

**¿Hay datos de clientes o empleados involucrados?**  
*(Implicaciones GDPR, DPA con proveedor, jurisdicción del procesamiento)*

*[Completad aquí]*